## Context Window: Why Chatbots Forget

Every model has a **context window limit** - the maximum number of tokens it can process in a single request.

This includes input + output tokens combined.

Let's see what happens when you exceed it!

In [ ]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY2"]
client = genai.Client(api_key=API_KEY)

In [ ]:
# Create a VERY long message to hit the limit
long_story = "Once upon a time, there was a programmer who loved Python. " * 100000

print(f"📏 Story length: {len(long_story):,} characters")
print(f"📏 Estimated tokens: ~{len(long_story) // 4:,}\n")

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=long_story,
        config={"max_output_tokens": 100}
    )
    print("✅ Request succeeded!")
    print(f"📊 Tokens used: {response.usage_metadata.total_token_count:,}")
    print(f"\n💡 Gemini 2.5 Flash has a HUGE context window (~1M tokens)")
    print(f"   So this request fits comfortably!")
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("\n💡 This is what happens when you exceed the context window!")

### What is Context Window?

**Context Window = Maximum tokens in a single request**

Different models have different limits:
- **Gemini 2.5 Flash**: ~1 million tokens
- **GPT-4**: ~128K tokens
- **Claude 4**: ~200K tokens




![image-2.png](attachment:image-2.png)


### Context Window in Conversations

In chat applications, the context window includes:
- **All previous messages** (entire conversation history)
- **Current message**
- **Response**

As conversations grow, tokens accumulate!

In [ ]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")

    return response.text

# Start conversation
print("👤: Hi! My name is Yash")
chat("Hi! My name is Yash")

print("👤: I'm 25 years old")
chat("I'm 25 years old")

print("👤: I love Python programming")
chat("I love Python programming")

print("👤: I work as a software engineer")
chat("I work as a software engineer")

print("👤: What's my name?")
chat("What's my name?")

### The Problem: Tokens Keep Growing

Notice how tokens increase with each message?

In a real long conversation:
- Message 1: 50 tokens
- Message 10: 500 tokens  
- Message 50: 2,500 tokens
- Message 100: 5,000 tokens
- Message 1000: 50,000 tokens

**Eventually, you'll hit the context window limit!**

### Solution: Remove Old Messages

When approaching the limit, remove oldest messages.

**This is why chatbots "forget"!**

In [ ]:
MAX_MESSAGES = 6  # Keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    # Remove old messages if too many
    if len(messages) > MAX_MESSAGES:
        removed = messages.pop(0)  # Remove oldest
        print(f"🗑️  Removed old message: {removed.parts[0].text[:50]}...\n")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    print(f"🤖: {response.text}")
    print(f"📊 Messages in memory: {len(messages)}\n")

    return response.text

# Reset and try again
messages = []

print("👤: My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("👤: I have a dog named Max")
chat_with_limit("I have a dog named Max")

print("👤: I live in Mumbai")
chat_with_limit("I live in Mumbai")

print("👤: I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("👤: What's my favorite color?")
chat_with_limit("What's my favorite color?")

### Key Takeaways

1. **Context Window** = Maximum tokens in a single request (input + output)

2. **In conversations**, tokens grow with each message

3. **Must remove old messages** when approaching limit

4. **This causes "forgetting"** - AI loses old context

5. **Exceeding limit = Error** - Request will fail

### Real-World Solutions

- **Summarize** old messages instead of removing
- **Store** important info separately (database)
- **Prioritize** recent messages
- **Use RAG** (Retrieval Augmented Generation) for long-term memory
- **Monitor token usage** and manage proactively